# PySpark Test - Haja Coração

Teste de processamento de dados de BPM usando PySpark em AWS S3

In [ ]:
# Iniciar Spark
from pyspark import SparkConf
from pyspark.sql import SparkSession

conf = SparkConf()
conf.set('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.2.0')
conf.set('spark.hadoop.fs.s3a.aws.credentials.provider', 'com.amazonaws.auth.InstanceProfileCredentialsProvider')
spark = SparkSession.builder.config(conf=conf).getOrCreate()

In [ ]:
# Carregar dados BPM de atletas do S3 Raw
# Atualizar o caminho conforme necessário
df_bpm = spark.read.option('delimiter', ',').option('header', 'true').csv('s3a://SEU-BUCKET-RAW/dados-teste.csv')

# Exibir dados
df_bpm.show()

In [ ]:
# Filtrar atletas com BPM anormal (>150 ou <40)
from pyspark.sql.functions import col

df_anormal = df_bpm.filter((col('bpm') > 150) | (col('bpm') < 40))
print(f"Atletas com BPM anormal: {df_anormal.count()}")
df_anormal.show()

In [ ]:
# Calcular média e desvio padrão de BPM
from pyspark.sql.functions import avg, stddev, col

stats = df_bpm.agg(
    avg('bpm').alias('media_bpm'),
    stddev('bpm').alias('desvio_bpm')
)
stats.show()

In [ ]:
# Salvar dados processados no S3 Trusted
output_path = 's3a://SEU-BUCKET-TRUSTED/processed_bpm'
df_bpm.write.mode('overwrite').parquet(output_path)
print(f"Dados salvos em: {output_path}")